# 🗄️ SQL & Bases de Données
**Cours couverts :** Introduction to SQL, Intermediate SQL, Introduction to Relational Databases in SQL, Joining Data in SQL, Database Design, Introduction to SQL Querying with AI

---
Ce notebook utilise SQLite en mémoire pour exécuter du SQL directement depuis Python.

In [ ]:
import sqlite3
import pandas as pd

# Créer une base de données SQLite en mémoire
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Fonction utilitaire pour afficher les résultats
def sql(query, conn=conn):
    return pd.read_sql_query(query, conn)

# Créer les tables de démonstration
cur.executescript("""
    CREATE TABLE pays (
        id INTEGER PRIMARY KEY,
        nom TEXT NOT NULL,
        continent TEXT,
        population INTEGER,
        pib_milliards REAL
    );

    CREATE TABLE villes (
        id INTEGER PRIMARY KEY,
        nom TEXT NOT NULL,
        pays_id INTEGER REFERENCES pays(id),
        population INTEGER,
        est_capitale INTEGER DEFAULT 0
    );

    CREATE TABLE employes (
        id INTEGER PRIMARY KEY,
        nom TEXT,
        departement TEXT,
        salaire REAL,
        manager_id INTEGER REFERENCES employes(id),
        date_embauche TEXT
    );

    INSERT INTO pays VALUES
        (1, 'France', 'Europe', 68000000, 2780.0),
        (2, 'Allemagne', 'Europe', 83000000, 4260.0),
        (3, 'Japon', 'Asie', 125000000, 4940.0),
        (4, 'Brésil', 'Amérique', 215000000, 1920.0),
        (5, 'USA', 'Amérique', 331000000, 25460.0);

    INSERT INTO villes VALUES
        (1, 'Paris', 1, 2161000, 1),
        (2, 'Lyon', 1, 522000, 0),
        (3, 'Berlin', 2, 3645000, 1),
        (4, 'Munich', 2, 1472000, 0),
        (5, 'Tokyo', 3, 13960000, 1),
        (6, 'São Paulo', 4, 12325000, 0),
        (7, 'Brasília', 4, 3094000, 1),
        (8, 'New York', 5, 8336000, 0),
        (9, 'Washington DC', 5, 705000, 1);

    INSERT INTO employes VALUES
        (1, 'Alice Martin', 'IT', 75000, NULL, '2019-03-15'),
        (2, 'Bob Dupont', 'IT', 62000, 1, '2021-06-01'),
        (3, 'Charlie Leroy', 'Marketing', 58000, 4, '2020-09-10'),
        (4, 'Diana Bernard', 'Marketing', 80000, NULL, '2018-01-20'),
        (5, 'Ethan Moreau', 'RH', 55000, NULL, '2022-04-05'),
        (6, 'Fiona Simon', 'IT', 90000, 1, '2017-07-22'),
        (7, 'Georges Petit', 'Finance', 70000, NULL, '2020-11-30'),
        (8, 'Hugo Lambert', 'Finance', 65000, 7, '2021-03-15');
""")
conn.commit()
print("Base de données créée avec succès !")

## 1. Requêtes SELECT de base

In [ ]:
# SELECT de base
sql("SELECT * FROM pays")

In [ ]:
# Colonnes spécifiques avec alias
sql("""
    SELECT nom AS pays,
           population / 1000000.0 AS pop_millions,
           pib_milliards
    FROM pays
    ORDER BY pib_milliards DESC
""")

In [ ]:
# WHERE avec conditions
sql("""
    SELECT nom, continent, pib_milliards
    FROM pays
    WHERE continent = 'Europe'
      AND pib_milliards > 3000
""")

In [ ]:
# LIKE, IN, BETWEEN, IS NULL
sql("SELECT * FROM employes WHERE nom LIKE 'A%'")

In [ ]:
sql("SELECT * FROM employes WHERE departement IN ('IT', 'Finance')")

In [ ]:
sql("SELECT * FROM employes WHERE salaire BETWEEN 60000 AND 80000")

In [ ]:
sql("SELECT * FROM employes WHERE manager_id IS NULL")  # les managers

## 2. Fonctions d'agrégation et GROUP BY

In [ ]:
# Fonctions d'agrégation : COUNT, SUM, AVG, MIN, MAX
sql("""
    SELECT
        COUNT(*) AS nb_employes,
        AVG(salaire) AS salaire_moyen,
        MIN(salaire) AS salaire_min,
        MAX(salaire) AS salaire_max,
        SUM(salaire) AS masse_salariale
    FROM employes
""")

In [ ]:
# GROUP BY avec HAVING
sql("""
    SELECT
        departement,
        COUNT(*) AS nb_employes,
        ROUND(AVG(salaire), 0) AS salaire_moyen
    FROM employes
    GROUP BY departement
    HAVING COUNT(*) >= 2
    ORDER BY salaire_moyen DESC
""")
# Rappel : WHERE filtre avant agrégation, HAVING filtre après

## 3. Jointures (JOIN)

In [ ]:
# INNER JOIN : seulement les enregistrements qui ont une correspondance
sql("""
    SELECT v.nom AS ville, p.nom AS pays, v.population
    FROM villes v
    INNER JOIN pays p ON v.pays_id = p.id
    ORDER BY v.population DESC
    LIMIT 5
""")

In [ ]:
# LEFT JOIN : tous les enregistrements de la table gauche
sql("""
    SELECT p.nom AS pays, v.nom AS capitale
    FROM pays p
    LEFT JOIN villes v ON p.id = v.pays_id AND v.est_capitale = 1
""")

In [ ]:
# SELF JOIN : jointure d'une table avec elle-même
# Ex : trouver chaque employé avec le nom de son manager
sql("""
    SELECT e.nom AS employe, m.nom AS manager
    FROM employes e
    LEFT JOIN employes m ON e.manager_id = m.id
    ORDER BY manager
""")

In [ ]:
# Jointure sur plusieurs tables
sql("""
    SELECT v.nom AS ville, p.nom AS pays, p.continent,
           v.population AS pop_ville, p.population AS pop_pays,
           ROUND(100.0 * v.population / p.population, 1) AS pct_pop
    FROM villes v
    JOIN pays p ON v.pays_id = p.id
    WHERE v.est_capitale = 1
    ORDER BY pct_pop DESC
""")

## 4. Sous-requêtes (Subqueries)

In [ ]:
# Sous-requête dans WHERE
sql("""
    SELECT nom, salaire
    FROM employes
    WHERE salaire > (SELECT AVG(salaire) FROM employes)
    ORDER BY salaire DESC
""")

In [ ]:
# Sous-requête dans FROM (table dérivée)
sql("""
    SELECT departement, salaire_moyen
    FROM (
        SELECT departement, ROUND(AVG(salaire), 0) AS salaire_moyen
        FROM employes
        GROUP BY departement
    ) AS stats_dept
    WHERE salaire_moyen > 65000
""")

In [ ]:
# EXISTS : vérifie si la sous-requête retourne des lignes
sql("""
    SELECT nom
    FROM employes e
    WHERE EXISTS (
        SELECT 1 FROM employes sub
        WHERE sub.manager_id = e.id
    )
""")  # Employes qui sont managers de quelqu'un

## 5. Fonctions de fenêtre (Window Functions)

In [ ]:
# ROW_NUMBER, RANK, DENSE_RANK
sql("""
    SELECT
        nom,
        departement,
        salaire,
        ROW_NUMBER() OVER (PARTITION BY departement ORDER BY salaire DESC) AS rang_dept,
        RANK()       OVER (ORDER BY salaire DESC) AS rang_global
    FROM employes
    ORDER BY departement, rang_dept
""")

In [ ]:
# LAG / LEAD : accéder à la ligne précédente/suivante
sql("""
    SELECT
        nom,
        salaire,
        LAG(salaire) OVER (ORDER BY salaire) AS salaire_precedent,
        salaire - LAG(salaire) OVER (ORDER BY salaire) AS difference
    FROM employes
    ORDER BY salaire
""")

In [ ]:
# SUM cumulatif avec OVER
sql("""
    SELECT
        nom,
        departement,
        salaire,
        SUM(salaire) OVER (PARTITION BY departement ORDER BY salaire
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS salaire_cumulatif
    FROM employes
    ORDER BY departement, salaire
""")

## 6. CTEs (Common Table Expressions)

In [ ]:
# CTE : requête nommée pour améliorer la lisibilité
sql("""
    WITH stats_departement AS (
        SELECT
            departement,
            AVG(salaire) AS salaire_moyen,
            MAX(salaire) AS salaire_max
        FROM employes
        GROUP BY departement
    ),
    employes_hauts_salaires AS (
        SELECT nom, departement, salaire
        FROM employes
        WHERE salaire > 70000
    )
    SELECT e.nom, e.departement, e.salaire, s.salaire_moyen
    FROM employes_hauts_salaires e
    JOIN stats_departement s ON e.departement = s.departement
    ORDER BY e.salaire DESC
""")

## 7. Conception de base de données (Database Design)

### Normalisation et relations

In [ ]:
# Les 3 types de relations :
# 1. One-to-One  : un pays a une seule capitale
# 2. One-to-Many : un pays a plusieurs villes
# 3. Many-to-Many : un employé peut avoir plusieurs projets, un projet plusieurs employés

# Pour une relation Many-to-Many, on utilise une TABLE DE JONCTION
cur.executescript("""
    CREATE TABLE projets (
        id INTEGER PRIMARY KEY,
        nom TEXT,
        budget REAL
    );

    CREATE TABLE employes_projets (
        employe_id INTEGER REFERENCES employes(id),
        projet_id  INTEGER REFERENCES projets(id),
        role TEXT,
        PRIMARY KEY (employe_id, projet_id)  -- clé primaire composite
    );

    INSERT INTO projets VALUES
        (1, 'Migration Cloud', 500000),
        (2, 'Refonte Site Web', 120000);

    INSERT INTO employes_projets VALUES
        (1, 1, 'Chef de projet'),
        (2, 1, 'Développeur'),
        (6, 1, 'Architecte'),
        (3, 2, 'Designer'),
        (4, 2, 'Chef de projet');
""")
conn.commit()

# Requête sur la table Many-to-Many
sql("""
    SELECT e.nom, p.nom AS projet, ep.role
    FROM employes_projets ep
    JOIN employes e ON ep.employe_id = e.id
    JOIN projets p ON ep.projet_id = p.id
    ORDER BY p.nom, e.nom
""")

### Formes normales (rappel)

In [ ]:
# Les formes normales en résumé :
notes = """
1NF (Première forme normale) :
  - Chaque cellule contient une seule valeur atomique
  - Pas de groupes répétitifs
  - Chaque ligne est unique (clé primaire)
  MAUVAIS: nom_produits = "Pomme, Banane, Cerise"
  BON: une ligne par produit

2NF (Deuxième forme normale) :
  - Doit être en 1NF
  - Chaque attribut non-clé dépend de TOUTE la clé primaire
  (Problème possible avec les clés composites)

3NF (Troisième forme normale) :
  - Doit être en 2NF
  - Pas de dépendances transitives
  MAUVAIS: commande(commande_id, client_id, ville_client)
            ville_client dépend de client_id, pas de commande_id
  BON: séparer en table clients et commandes
"""
print(notes)

## 8. CASE WHEN et fonctions de chaînes/dates

In [ ]:
# CASE WHEN : logique conditionnelle en SQL
sql("""
    SELECT
        nom,
        salaire,
        CASE
            WHEN salaire < 60000 THEN 'Junior'
            WHEN salaire < 75000 THEN 'Confirmé'
            ELSE 'Senior'
        END AS niveau,
        UPPER(departement) AS dept_majuscules,
        LENGTH(nom) AS longueur_nom,
        SUBSTR(date_embauche, 1, 4) AS annee_embauche
    FROM employes
    ORDER BY salaire
""")

## Exercices de révision SQL

In [ ]:
# EXERCICE 1 : Trouver les 2 villes les plus peuplées par continent
# (utiliser une fenêtre ROW_NUMBER)
# Hint : JOIN villes et pays, puis window function

# À compléter !
# sql("""
# ...votre requête...
# """)

In [ ]:
# EXERCICE 2 : Pour chaque département, montrer l'employé avec le salaire le plus haut
# et l'écart par rapport à la moyenne du département

# À compléter !

In [ ]:
# --- SOLUTIONS ---

# Ex 1 : Top 2 villes par continent
sql("""
    WITH classement AS (
        SELECT v.nom AS ville, p.continent, v.population,
               ROW_NUMBER() OVER (PARTITION BY p.continent ORDER BY v.population DESC) AS rang
        FROM villes v
        JOIN pays p ON v.pays_id = p.id
    )
    SELECT ville, continent, population
    FROM classement
    WHERE rang <= 2
    ORDER BY continent, rang
""")

In [ ]:
# Ex 2 : Employé le plus payé par département + écart à la moyenne
sql("""
    WITH stats AS (
        SELECT
            departement,
            MAX(salaire) AS salaire_max,
            ROUND(AVG(salaire), 0) AS salaire_moyen
        FROM employes
        GROUP BY departement
    )
    SELECT e.nom, e.departement, e.salaire,
           s.salaire_moyen,
           ROUND(e.salaire - s.salaire_moyen, 0) AS ecart
    FROM employes e
    JOIN stats s ON e.departement = s.departement AND e.salaire = s.salaire_max
    ORDER BY e.salaire DESC
""")